# VWAP: data → model research → trading → both results

The selection objective is **set 1 net trading P&L**, with fixed execution assumptions. Set 2 never enters calibration. The same final model is used throughout both replays. Extract the full project before running.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
import research as r
b1, trades1, _ = r.load("data")
display(b1)
display(trades1)

,timestamp,time,bid1_px,bid1_qty,bid1_age_s,bid2_px,bid2_qty,bid2_age_s,bid3_px,bid3_qty,bid3_age_s,ask1_px,ask1_qty,ask1_age_s,ask2_px,ask2_qty,ask2_age_s,ask3_px,ask3_qty,ask3_age_s,mid,spread,flow,trade_qty
0,T01,09:00:00,8.50,10,30,8.45,50,30,8.40,24,30,8.55,10,30,8.60,300,30,8.65,300,30,8.525,0.05,0.0,0.0
1,T02,09:00:05,8.45,50,35,8.40,24,35,8.35,100,35,8.55,10,35,8.60,300,35,8.65,300,35,8.500,0.10,-1.0,10.0
2,T03,09:00:10,8.45,50,40,8.40,24,40,8.35,100,40,8.50,20,0,8.55,5,40,8.60,300,40,8.475,0.05,1.0,5.0
3,T04,09:00:15,8.45,10,45,8.40,24,45,8.35,100,45,8.50,20,5,8.55,5,45,8.60,300,45,8.475,0.05,-1.0,40.0
4,T05,09:00:20,8.40,24,50,8.35,100,50,8.30,60,0,8.45,30,0,8.50,20,10,8.55,5,50,8.425,0.05,-1.0,10.0
5,T06,09:00:25,8.40,80,0,8.35,100,55,8.30,60,5,8.45,27,5,8.50,20,15,8.55,5,55,8.425,0.05,1.0,3.0
6,T07,09:00:30,8.40,580,0,8.35,100,60,8.30,60,10,8.45,27,10,8.50,20,20,8.55,5,60,8.425,0.05,0.0,0.0
7,T08,09:00:35,8.40,580,5,8.35,100,65,8.30,60,15,8.45,40,0,8.50,20,25,8.55,5,65,8.425,0.05,1.0,27.0
8,T09,09:00:40,8.40,80,10,8.35,100,70,8.30,60,20,8.45,25,5,8.50,20,30,8.55,5,70,8.425,0.05,1.0,15.0
9,T10,09:00:45,8.40,75,15,8.35,100,75,8.30,60,25,8.45,25,10,8.50,20,35,8.55,5,75,8.425,0.05,-1.0,5.0


,interval,printed_before,seq,aggressor,price,qty,buyer,seller,signed_qty
0,T01->T02,T02,1,SELL,8.50,10,CP-C,CP-B,-10
1,T02->T03,T03,1,BUY,8.55,5,CP-B,CP-C,5
2,T03->T04,T04,1,SELL,8.45,40,CP-C,CP-A,-40
3,T04->T05,T05,1,SELL,8.45,10,CP-C,CP-B,-10
4,T05->T06,T06,1,BUY,8.45,3,CP-B,CP-C,3
5,T07->T08,T08,1,BUY,8.45,27,CP-B,CP-C,27
6,T08->T09,T09,1,BUY,8.45,15,CP-B,CP-D,15
7,T09->T10,T10,1,SELL,8.40,5,CP-C,CP-B,-5
8,T10->T11,T11,1,BUY,8.45,25,CP-A,CP-D,25
9,T10->T11,T11,2,BUY,8.50,20,CP-A,CP-C,20


## Models

Direct crossed model:
$$V^{(1)}=\frac{\sum_i(q_i^a b_i+q_i^b a_i)}{\sum_i(q_i^a+q_i^b)}.$$

For weighted side averages, use $B_i=\ell_iq_i^bd(age_i^b)$ and $A_i=\ell_iq_i^ad(age_i^a)$. With $D_b=\sum_iB_i,D_a=\sum_iA_i$,
$$V=\frac{D_a\bar b+D_b\bar a}{D_a+D_b},\quad\bar b=\frac{\sum_iB_ib_i}{D_b},\quad\bar a=\frac{\sum_iA_ia_i}{D_a}.$$

Then add only available flow: $\widehat m_{t+1}=V_t+\gamma F_ts_t$.

## VWAP research before trading

Compare the three fixed VWAP definitions with midpoint, using the next midpoint as target and no execution assumptions. These are prediction diagnostics. The trading rule and P&L-based parameter selection follow afterwards.

In [2]:
references = {"Midpoint": {"kind":"mid"}, "VWAP 1": {"kind":"direct"}, "VWAP 2": {"kind":"weighted","w2":.25,"w3":.0625,"decay":"none"}, "VWAP 3": {"kind":"weighted","w2":.25,"w3":.0625,"decay":"exp","tau":30}}
display(pd.DataFrame([{ "model": name, **r.metric(b1,r.predict(b1,model))} for name,model in references.items()]))

,model,mae,rmse,bias
0,Midpoint,0.023684,0.051936,0.002632
1,VWAP 1,0.046540,0.059771,0.004732
2,VWAP 2,0.041631,0.060044,0.007782
3,VWAP 3,0.037261,0.058572,0.005136


## Executable rule

Search both directions and quantities 1–50 against actual displayed depth. Trade only when the maximum surplus is positive:
$$G_t(d,Q)=\sum_i x_i d(\widehat m_{t+1}-p_i)-0.20-0.12s_tQ.$$
Inventory is limited to 100 units and marked notional to £5,000. T20 liquidates the full position. Cash is updated with decimal arithmetic:
$$C_t=C_{t-1}-d\sum_i x_ip_i-f,\quad I_t=I_{t-1}+dQ.$$
Only when terminal inventory is zero is final cash realised net P&L.

## Select model parameters using the defined trading rule

The execution policy is now fixed. Choose parameters using net P&L on set 1. Prediction error remains a diagnostic.

In [3]:
stages, search, selected = r.calibrate(b1)
display(pd.DataFrame(stages)[["name","model","net_pnl","fees","orders","mae"]])
frozen = {"model": selected["model"], "execution": r.EXECUTION}
print(json.dumps(frozen, indent=2))

{
  "model": {
    "kind": "weighted",
    "w2": 1.0,
    "w3": 0.0625,
    "decay": "none",
    "gamma": -0.5
  },
  "execution": {
    "fee": 0.2,
    "clip": 50,
    "position_limit": 100,
    "notional_limit": 5000.0,
    "buffer": 0.12
  }
}


,name,model,net_pnl,fees,orders,mae
0,Midpoint,{'kind': 'mid'},0.00,0.0,0,0.023684
1,VWAP 1,{'kind': 'direct'},1.70,2.0,10,0.046540
2,VWAP 2 fixed,"{'kind': 'weighted', 'w2': 0.25, 'w3': 0.0625,...",0.35,0.4,2,0.041631
3,VWAP 3 fixed,"{'kind': 'weighted', 'w2': 0.25, 'w3': 0.0625,...",-2.80,0.4,2,0.037261
4,Geometric tuned,"{'kind': 'weighted', 'w2': 0.25, 'w3': 0.0625,...",0.35,0.4,2,0.041631
5,Independent tuned,"{'kind': 'weighted', 'w2': 1.0, 'w3': 0.0625, ...",4.50,1.0,5,0.046985
6,Age tuned,"{'kind': 'weighted', 'w2': 1.0, 'w3': 0.0625, ...",4.50,1.0,5,0.046985
7,Flow tuned,"{'kind': 'weighted', 'w2': 1.0, 'w3': 0.0625, ...",6.25,1.6,8,0.047927


## Research decision

The geometric search retains the starting weights. Independent weights improve set 1 net P&L to £4.50. Age decay does not improve it and is rejected. Completed trade flow raises training net P&L to £6.25. The final weights are $(1,1,0.0625)$, no decay, and $\gamma=-0.5$. This is an in-sample profit fit; MAE is diagnostic. The HTML includes every candidate and all charts.

In [4]:
b2, trades2, _ = r.load("data", test=True)
results = {}
for name, book in [("Set 1",b1),("Set 2",b2)]:
    log, fills, summary = r.simulate(book, r.predict(book, frozen["model"]), **frozen["execution"])
    r.audit(book, log, fills, summary)
    results[name] = summary
    print(name)
    display(log.loc[log.qty>0,["timestamp","action","qty","avg_price","fee","position","cash","equity"]])
    display(fills)
display(pd.DataFrame(results).T)

Set 1
Set 2


,timestamp,action,qty,avg_price,fee,position,cash,equity
4,T05,BUY,30,8.450000,0.2,30,-253.70,-0.950
6,T07,BUY,27,8.450000,0.2,57,-482.05,-1.825
9,T10,BUY,25,8.450000,0.2,82,-693.50,-2.650
10,T11,SELL,35,8.521429,0.2,47,-395.45,7.575
11,T12,SELL,40,8.550000,0.2,7,-53.65,6.375
12,T13,SELL,50,8.550000,0.2,-43,373.65,4.925
16,T17,BUY,10,8.650000,0.2,-33,286.95,2.325
19,T20,LIQUIDATE,33,8.500000,0.2,0,6.25,6.250


,timestamp,order,action,side,level,qty,price,signed_qty,cash_delta
0,T05,1,BUY,BUY,1,30,8.45,30,-253.50
1,T07,2,BUY,BUY,1,27,8.45,27,-228.15
2,T10,3,BUY,BUY,1,25,8.45,25,-211.25
3,T11,4,SELL,SELL,1,15,8.55,-15,128.25
4,T11,4,SELL,SELL,2,20,8.50,-20,170.00
5,T12,5,SELL,SELL,1,40,8.55,-40,342.00
6,T13,6,SELL,SELL,1,50,8.55,-50,427.50
7,T17,7,BUY,BUY,1,10,8.65,10,-86.50
8,T20,8,LIQUIDATE,BUY,1,33,8.50,33,-280.50


,timestamp,action,qty,avg_price,fee,position,cash,equity
3,T04,BUY,30,8.50,0.2,30,-255.2,-0.95
4,T05,BUY,30,8.50,0.2,60,-510.4,-1.90
9,T10,SELL,20,8.65,0.2,40,-337.6,9.40
12,T13,SELL,30,8.55,0.2,10,-81.3,4.45
19,T20,LIQUIDATE,10,8.45,0.2,0,3.0,3.00


,timestamp,order,action,side,level,qty,price,signed_qty,cash_delta
0,T04,1,BUY,BUY,1,30,8.50,30,-255.0
1,T05,2,BUY,BUY,1,30,8.50,30,-255.0
2,T10,3,SELL,SELL,1,20,8.65,-20,173.0
3,T13,4,SELL,SELL,1,30,8.55,-30,256.5
4,T20,5,LIQUIDATE,SELL,1,10,8.45,-10,84.5


,net_pnl,gross_pnl,fees,orders,filled_units,terminal_position,peak_notional,max_drawdown
Set 1,6.25,7.85,1.6,8.0,250.0,0.0,690.85,5.25
Set 2,3.00,4.00,1.0,5.0,120.0,0.0,514.50,6.40


## Result and interpretation

Set 1: **£7.85 gross − £1.60 fees = £6.25 net**. Set 2: **£4.00 gross − £1.00 fees = £3.00 net**. Both finish flat. The same strategy and execution constants are used for both. Set 1 was optimised, and set 2 is a reused holdout from earlier work. Neither result proves durable live profitability. The simulation assumes immediate displayed liquidity and no effect on future recorded books.

In [5]:
import subprocess, sys
completed = subprocess.run([sys.executable,"-m","unittest","-v","test_research.py"],check=True,capture_output=True,text=True)
print(completed.stderr)

test_decay_and_zero_flow_invariance (test_research.AuditTests.test_decay_and_zero_flow_invariance) ... ok
test_depth_example (test_research.AuditTests.test_depth_example) ... ok
test_direct_formula_independently (test_research.AuditTests.test_direct_formula_independently) ... ok
test_flow_alignment (test_research.AuditTests.test_flow_alignment) ... ok
test_future_features_do_not_change_past (test_research.AuditTests.test_future_features_do_not_change_past) ... ok
test_midpoint_and_incomplete_liquidation (test_research.AuditTests.test_midpoint_and_incomplete_liquidation) ... ok
test_original_reference_mae (test_research.AuditTests.test_original_reference_mae) ... ok
test_selection_and_both_cash_ledgers (test_research.AuditTests.test_selection_and_both_cash_ledgers) ... ok

----------------------------------------------------------------------
Ran 8 tests in 2.120s

OK



## Regenerate the report

Run `python research.py --data data --out results`. The HTML uses embedded KaTeX, fonts and Plotly and works offline. The notebook and HTML use the same numerical implementation.

## Extensions: midpoint blends, free weights and aggressive sizing

The last-midpoint forecast is $m_t$. A blend is $\alpha\hat m^{model}_{t+1}+(1-\alpha)m_t$. We also search independent weights $(1,w_1,w_2)$, vary the fee and spread buffer in the decision score, and increase the order-search clip while retaining the marked-notional and position limits.

In [ ]:
from pathlib import Path
import pandas as pd
for dataset in ["set_1","set_2"]:
    ext=pd.read_csv(Path("results")/(dataset+"_extensions.csv"))
    display(ext[ext.model.str.contains("Last|Blend|Policy|Clip")][["model","mae","net_pnl","fees","orders","filled_units"]])

## Unconstrained weights

Search positive $(1,w_1,w_2)$ weights without w2≤w1≤1.

In [ ]:
pd.read_csv("results/unconstrained_weights.csv").sort_values("set1_net_pnl",ascending=False).head(20)